# CS604 Colab launcher — Ana: random vs aimed training crops (pilot scale)

Does it matter whether the training crops contain the organism? Arms in `configs/ablation_train_crops.yaml`, on the same 1,000-class pilot subset and holdout as the earlier ablations, base = LoRA r24 without extras (CodaBench 0.784, 4.12%). Only the training crop changes:

| Arm | Training crops |
|---|---|
| A_random_crops | today's random crops, 8–100% of the photo (timm default) |
| S25_bigger_crops | random crops of at least 25% of the photo ("just bigger", the control) |
| BOX_aimed_crops | random crops that must contain the organism box ("aimed") |

The boxes come from `scripts/find_boxes.py`: DDT co-localization (Wei et al.). For each class, the pretrained backbone's patch features from all its training images are compared, and what they share (the organism) gives a score map and a box per image. Nothing is trained for this, and only the pilot's fit images are used, never the holdout.

Three phases: (1) A and S25; (2) find the boxes, with diagnostics (how often today's crops miss the organism, whether box crops are recognised better than random crops of the same size, and an overlay figure); (3) BOX. Finished arms and boxes are reused if you Run all again.

About 45 min on an A100: data ~15 min, prototypes ~5 min, 3 arms × ~6 min, boxes ~5 min. Results go to `CS604_RUNS/ablation_ablation_train_crops/`. The box-finding time is printed and saved in `boxes_stats.json`: report it as data preparation, like prototype building.

Runtime: **A100 GPU**. Use **Runtime → Run all**.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
METHOD = "lora_r24_336_no_extras"  # base config; only used to build prototypes_336.pt if it is missing
PROTO_PT = f"{DRIVE_OUT}/prototypes_336.pt"
SPEC = "configs/ablation_train_crops.yaml"
PILOT_DATA_DIR = "/content/pilot_data"
FEATURE_DIR = "/content/features_pilot"  # local disk
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 4b. Build class prototypes at 336 (once)

Same as Van's prototype head, but built from 336 features (built with `configs/{METHOD}.yaml`, so at its `img_size`). Run **once**; about 20–25 min on an A100. Takes a while (embeds train images). Skip if the file already exists on Drive.

In [ ]:
from pathlib import Path

if Path(PROTO_PT).is_file():
    print("Already have", PROTO_PT)
else:
    !python scripts/build_prototypes.py --config configs/{METHOD}.yaml --data_root {DATA_DIR} --output {PROTO_PT}

## 5. Phase 1: A and S25 (no boxes needed)

In [ ]:
import subprocess
import sys
from pathlib import Path

OUT_DIR = Path(DRIVE_OUT) / f"ablation_{Path(SPEC).stem}"
PILOT_ROOT = f"{PILOT_DATA_DIR}/c1000_h5_s0"  # built by scripts/ablation.py from the spec's pilot block


def run(cmd, what):
    proc = subprocess.run([sys.executable, "-u", *cmd])
    if proc.returncode != 0:
        end_runtime(f"{what} failed (exit {proc.returncode}); see the output above")
        raise SystemExit(f"{what} failed")


def ablation(*only):
    cmd = ["scripts/ablation.py", "--spec", SPEC, "--data_root", DATA_DIR, "--output_dir", DRIVE_OUT,
           "--pilot_data_dir", PILOT_DATA_DIR, "--feature_dir", FEATURE_DIR]
    run(cmd + (["--only", *only] if only else []), "ablation")

ablation("A_random_crops", "S25_bigger_crops")


## 6. Phase 2: find the organism boxes (DDT co-localization) and check them

Pilot fit images only. Prints how often today's random crops cover under half of the box, the class-prototype top-1 of box crops vs random crops of the same size vs the whole image, and the box-finding GPU time.

In [ ]:
import json

from IPython.display import Image, display

if not (OUT_DIR / "boxes.pt").is_file():
    run(["scripts/find_boxes.py", "--config", f"configs/{METHOD}.yaml",
         "--data_root", PILOT_ROOT, "--out_dir", str(OUT_DIR)], "box finding")
print(json.dumps(json.loads((OUT_DIR / "boxes_stats.json").read_text()), indent=2))
display(Image(filename=str(OUT_DIR / "boxes_overlay.png")))


## 7. Phase 3: BOX arm (A and S25 are reused)

In [ ]:
ablation()


## 8. Results

`delta_vs_A_points`: pilot holdout top-1 against A, in percentage points. Treat under ~0.3 as a tie (one seed).

- **BOX vs A**: does aiming the training crops at the organism help?
- **BOX vs S25**: aimed crops vs just bigger crops.
- With `boxes_stats.json`: if A's crops often miss the organism but BOX does not help, the model tolerates noisy crops.


In [ ]:
import pandas as pd

table = pd.read_csv(OUT_DIR / "ablation.csv")
display(table[["arm", "final_val_acc", "delta_vs_A_points", "best_val_acc", "early_dip",
               "backbone_pct", "training_min", "peak_vram_gb", "wall_min"]])


## 9. End the runtime


In [ ]:
end_runtime("all cells finished; ending the runtime")